In [1]:
import pandas as pd

df = pd.read_csv("../../datasets/Bengaluru_House_Data_Cleaned.csv")

In [2]:
df.head()

,area_type,availability,location,size,total_sqft,bath,balcony,price,bhk,price_per_sqft
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,1056.0,2.0,1.0,39.07,2,3699.810606
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,2600.0,5.0,3.0,120.00,4,4615.384615
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,1440.0,2.0,3.0,62.00,3,4305.555556
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,1521.0,3.0,1.0,95.00,3,6245.890861
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,1200.0,2.0,1.0,51.00,2,4250.000000


In [3]:
df.shape

(13200, 10)

In [6]:
df["sqft_per_bhk"].describe()

count    13200.000000
mean       573.847262
std        388.079980
min          0.250000
25%        473.000000
50%        552.000000
75%        625.000000
max      26136.000000
Name: sqft_per_bhk, dtype: float64

In [ ]:
df.columns

Index(['area_type', 'availability', 'location', 'size', 'total_sqft', 'bath',
       'balcony', 'price', 'bhk', 'price_per_sqft'],
      dtype='object')

In [ ]:
df["sqft_per_bhk"] = df["total_sqft"] / df["bhk"]

In [ ]:
df.columns

Index(['area_type', 'availability', 'location', 'size', 'total_sqft', 'bath',
       'balcony', 'price', 'bhk', 'price_per_sqft', 'sqft_per_bhk'],
      dtype='object')

In [ ]:
df["sqft_per_bhk"].describe()

count    13200.000000
mean       573.847262
std        388.079980
min          0.250000
25%        473.000000
50%        552.000000
75%        625.000000
max      26136.000000
Name: sqft_per_bhk, dtype: float64

In [5]:
df["sqft_per_bhk"] = df["total_sqft"] / df["bhk"]

In [ ]:
df["sqft_per_bhk"].describe()

count    13200.000000
mean       573.847262
std        388.079980
min          0.250000
25%        473.000000
50%        552.000000
75%        625.000000
max      26136.000000
Name: sqft_per_bhk, dtype: float64

In [ ]:
df[df["sqft_per_bhk"] < 300].shape

(744, 11)

In [ ]:
df[df["sqft_per_bhk"] < 300].head(10)

,area_type,availability,location,size,total_sqft,bath,balcony,price,bhk,price_per_sqft,sqft_per_bhk
9,Plot Area,Ready To Move,Gandhi Bazar,6 Bedroom,1020.0,6.0,2.0,370.0,6,36274.509804,170.000000
45,Plot Area,Ready To Move,HSR Layout,8 Bedroom,600.0,9.0,2.0,200.0,8,33333.333333,75.000000
57,Plot Area,Ready To Move,Murugeshpalya,6 Bedroom,1407.0,4.0,1.0,150.0,6,10660.980810,234.500000
67,Plot Area,Ready To Move,Devarachikkanahalli,8 Bedroom,1350.0,7.0,0.0,85.0,8,6296.296296,168.750000
69,Plot Area,Ready To Move,Double Road,3 Bedroom,500.0,3.0,2.0,100.0,3,20000.000000,166.666667
77,Built-up Area,Ready To Move,Kaval Byrasandra,2 BHK,460.0,1.0,0.0,22.0,2,4782.608696,230.000000
87,Plot Area,Ready To Move,Rajaji Nagar,6 Bedroom,710.0,6.0,3.0,160.0,6,22535.211268,118.333333
117,Plot Area,Ready To Move,Hennur Road,2 Bedroom,276.0,3.0,3.0,23.0,2,8333.333333,138.000000
127,Plot Area,Ready To Move,Vishwapriya Layout,7 Bedroom,950.0,7.0,0.0,115.0,7,12105.263158,135.714286
147,Plot Area,Ready To Move,Dinnur,6 Bedroom,1034.0,5.0,2.0,185.0,6,17891.682785,172.333333


In [ ]:
df = df[df["sqft_per_bhk"] >= 300]

In [ ]:
df.shape

(12456, 11)

In [ ]:
len(df["location"].unique())

1215

In [ ]:
location_stats = df.groupby("location")["location"].count()
location_stats.sort_values(ascending=False)

location
Whitefield                    531
Sarjapur  Road                388
Electronic City               293
Kanakpura Road                262
Thanisandra                   231
                             ... 
Whitefield ECC Road             1
Williams Town                   1
Xavier Layout                   1
Yelahanka,MVIT college          1
Yemlur, Old Airport Road,       1
Name: location, Length: 1215, dtype: int64

In [ ]:
len(location_stats[location_stats <= 10])

993

In [ ]:
df = df[df["sqft_per_bhk"] >= 300].copy()

In [ ]:
location_stats_less_than_10 = location_stats[location_stats <= 10]

df["location"] = df["location"].apply(
    lambda x: "other" if x in location_stats_less_than_10 else x
)

In [ ]:
df["location"].nunique()

223

In [ ]:
import numpy as np

def remove_pps_outliers(df):
    df_out = pd.DataFrame()

    for key, subdf in df.groupby("location"):
        m = np.mean(subdf.price_per_sqft)
        st = np.std(subdf.price_per_sqft)

        reduced_df = subdf[
            (subdf.price_per_sqft >= (m - st)) &
            (subdf.price_per_sqft <= (m + st))
        ]

        df_out = pd.concat([df_out, reduced_df], ignore_index=True)

    return df_out

In [ ]:
df = remove_pps_outliers(df)

In [ ]:
df.shape

(10272, 11)

In [ ]:
def remove_bhk_outliers(df):
    exclude_indices = np.array([])

    for location, location_df in df.groupby("location"):

        bhk_stats = {}

        for bhk, bhk_df in location_df.groupby("bhk"):
            bhk_stats[bhk] = {
                'mean': np.mean(bhk_df.price_per_sqft),
                'std': np.std(bhk_df.price_per_sqft),
                'count': bhk_df.shape[0]
            }

        for bhk, bhk_df in location_df.groupby("bhk"):
            stats = bhk_stats.get(bhk - 1)

            if stats and stats['count'] > 5:
                exclude_indices = np.append(
                    exclude_indices,
                    bhk_df[bhk_df.price_per_sqft < stats['mean']].index.values
                )

    return df.drop(exclude_indices, axis='index')

In [ ]:
df = remove_bhk_outliers(df)

In [ ]:
df.shape

(7280, 11)

In [ ]:
df.to_csv("../../datasets/Bengaluru_House_Data_Final.csv", index=False)